In [1]:
# STANDIN A0
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-For-Cybersecurity-Lab-4.2-Robustness-attacks


train 267,984 rows   attack rate 0.1507
val    89,328 rows   attack rate 0.1507
test   89,329 rows   attack rate 0.1507


68 features: 58 continuous, 10 flag columns
Flag columns: Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count
  tree    loaded from models/tree.joblib
  logreg  loaded from models/logreg.joblib


  forest  loaded from models/forest.joblib


ATTACK_IDX: 50 detected attacks, forest probability 1.000-1.000
  attack types: {'DDoS': 25, 'DoS Hulk': 22, 'DoS GoldenEye': 2, 'SSH-Patator': 1}
EXPLAIN_IDX: 500 random test flows, 68 of them attacks
Setup complete: 20 shared names ready. The models are scored in step A2.


<!-- STEP C1 -->
## Part C: Break the model on purpose

### Step C1: Sort the features and set a budget

An evasion attack is only honest if the attacker changes things a real attacker can change (lab
section 3.4). The attacker sends the **forward** traffic, so they can pad packets and wait longer. They
do not control the victim's replies, and they cannot finish the attack with fewer packets than it
needs. Every feature therefore goes into one of three groups:

| Group | Meaning | Why |
|---|---|---|
| **Free** | timing (inter-arrival times, idle and active timers, flow duration) and forward packet / header / segment sizes | Waiting longer and padding packets cost the attacker almost nothing |
| **Costly** | forward packet and byte counts (`Total Fwd Packets`, `Total Length of Fwd Packets`, forward subflows, forward data packets) | Possible, but every extra packet is more work and makes the attack louder |
| **Fixed** | everything the responder produces (all backward features, its initial window), all flag counts, and every feature we are unsure about | The victim's network stack produces these; the attacker cannot touch them |

The groups are assigned with **name rules in code**, not by hand, in this order: backward or flag →
Fixed; timing → Free; forward sizes → Free; forward counts → Costly; anything else → Fixed. When in
doubt, Fixed: an attack that is weaker than reality is a safer mistake than one that lets the attacker
do the impossible.

Our data has no `Destination Port` or `Protocol` (Lab 1 removed them as ID columns), so those two
Fixed features from the lab text do not appear.

In [2]:
# STEP C1
FREE_SIZES = ["Fwd Header Length", "Avg Fwd Segment Size", "min_seg_size_forward"]
COSTLY_COUNTS = ["Total Fwd Packets", "Total Length of Fwd Packets", "Subflow Fwd Packets",
                 "Subflow Fwd Bytes", "act_data_pkt_fwd"]


def assign_group(name):
    """Free / Costly / Fixed from the feature name, plus the rule that decided it.

    The order matters: the backward check comes first, otherwise 'Bwd IAT Mean' would be Free
    because it contains 'IAT'.
    """
    if "Bwd" in name or "backward" in name.lower():
        return "Fixed", "backward direction: produced by the victim"
    if "Flag" in name:
        return "Fixed", "flag: set by the network stack / responder"
    if any(key in name for key in ("IAT", "Idle", "Active")) or name == "Flow Duration":
        return "Free", "timing: the attacker can wait longer"
    if name.startswith("Fwd Packet Length") or name in FREE_SIZES:
        return "Free", "forward size: the attacker can pad packets"
    if name in COSTLY_COUNTS:
        return "Costly", "forward count: more packets, a louder attack"
    return "Fixed", "unsure (mixes both directions or is derived): Fixed to be safe"


_assigned = {name: assign_group(name) for name in FEATURES}
GROUP = {name: group for name, (group, _) in _assigned.items()}
GROUP_RULE = {name: rule for name, (_, rule) in _assigned.items()}

assert set(GROUP) == set(FEATURES)
counts = pd.Series(GROUP).value_counts().reindex(["Free", "Costly", "Fixed"])
print(counts.to_string())
assert counts.to_dict() == {"Free": 25, "Costly": 5, "Fixed": 38}
for group in ("Free", "Costly", "Fixed"):
    names = [f for f in FEATURES if GROUP[f] == group]
    print(f"\n{group} ({len(names)}):\n  " + ", ".join(names))

Free      25
Costly     5
Fixed     38

Free (25):
  Flow Duration, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Fwd Header Length, Avg Fwd Segment Size, min_seg_size_forward, Active Mean, Active Std, Active Max, Active Min, Idle Mean, Idle Std, Idle Max, Idle Min

Costly (5):
  Total Fwd Packets, Total Length of Fwd Packets, Subflow Fwd Packets, Subflow Fwd Bytes, act_data_pkt_fwd

Fixed (38):
  Total Backward Packets, Total Length of Bwd Packets, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Fwd URG Flags, Bwd Header Length, Fwd Packets/s, Bwd Packets/s, Min Packet Length, Max Packet Length, Packet Length Mean, Packet Length Std, Packet Length Variance, FIN

In [3]:
# STEP C1
# The Fixed features that are Fixed only because we were unsure, with the reason.
unsure = [f for f in FEATURES if GROUP_RULE[f].startswith("unsure")]
print(f"{len(unsure)} features are Fixed by the 'when unsure, Fixed' rule:")
for f in unsure:
    print(f"  {f}")

11 features are Fixed by the 'when unsure, Fixed' rule:
  Flow Bytes/s
  Flow Packets/s
  Fwd Packets/s
  Min Packet Length
  Max Packet Length
  Packet Length Mean
  Packet Length Std
  Packet Length Variance
  Down/Up Ratio
  Average Packet Size
  Init_Win_bytes_forward


<!-- STEP C1 -->
**Why the "unsure" features are Fixed.** They fall in two kinds.

- *Mixed direction*: `Min/Max Packet Length`, `Packet Length Mean/Std/Variance`, `Average Packet Size`
  and `Down/Up Ratio` combine the attacker's packets with the victim's replies. The attacker can move
  only part of them, and not reliably.
- *Derived rates*: `Flow Bytes/s`, `Flow Packets/s` and `Fwd Packets/s` are computed from counts and
  duration. Waiting longer *lowers* them, so the attacker cannot raise them independently.

`Init_Win_bytes_forward` is set by the attacker's own TCP stack and could arguably be Costly. We keep
it Fixed for now; if C2's constrained attack evades nothing, it is the first candidate to move (the lab
allows this, as long as the report says so).

<!-- STEP C1 -->
### The constraint check and the budget

- `can_change(feature, old_value, new_value)` is `True` only when the feature is Free or Costly **and**
  the new value is at least the old one. The attacker can add packets and wait longer, never remove or
  speed up.
- **Budget:** one step raises one feature by at most **half its training standard deviation**, and at
  most **five features** may change in total.

In [4]:
# STEP C1
STEP_SIZE = 0.5 * TRAIN_STD          # pandas Series: the largest single-step increase per feature
MAX_CHANGES = 5                      # at most five different features may change per flow
CHANGEABLE = [f for f in FEATURES if GROUP[f] in ("Free", "Costly")]


def can_change(feature, old_value, new_value):
    """True only if the attacker may move this feature from old_value to new_value."""
    return GROUP[feature] in ("Free", "Costly") and new_value >= old_value


# Tests: no Fixed feature can ever change; nothing can decrease; Free and Costly can increase.
for f in FEATURES:
    if GROUP[f] == "Fixed":
        assert not can_change(f, 0.0, 1.0) and not can_change(f, 5.0, 5.0)
    else:
        assert can_change(f, 0.0, STEP_SIZE[f]) and can_change(f, 3.0, 3.0)
        assert not can_change(f, 1.0, 0.0)
print(f"can_change tests passed: {len(CHANGEABLE)} changeable features, "
      f"{len(FEATURES) - len(CHANGEABLE)} Fixed, none of which can change")

can_change tests passed: 30 changeable features, 38 Fixed, none of which can change


<!-- STEP C1 -->
### What one step means in real units

Part B showed that the standard deviations of our columns are inflated by outliers. The step size is
half a standard deviation, so it is worth seeing what that means for a typical flow before running the
attack: a "small" step can be huge compared with the median.

In [5]:
# STEP C1
def unit(name):
    """Unit of a changeable feature, for reading the table below."""
    if any(key in name for key in ("IAT", "Idle", "Active", "Duration")):
        return "µs"
    if "Length" in name or "Bytes" in name or "Size" in name or "seg_size" in name:
        return "bytes"
    return "packets"


steps = pd.DataFrame({
    "group": [GROUP[f] for f in CHANGEABLE],
    "unit": [unit(f) for f in CHANGEABLE],
    "train_median": TRAIN_MEDIAN[CHANGEABLE],
    "step_size": STEP_SIZE[CHANGEABLE],
}, index=pd.Index(CHANGEABLE, name="feature"))
steps["step_over_median"] = steps["step_size"] / steps["train_median"].abs().replace(0, np.nan)
print(steps.sort_values("step_over_median", ascending=False)
      .to_string(float_format=lambda v: f"{v:,.1f}"))

                              group     unit  train_median    step_size  step_over_median
feature                                                                                  
Fwd IAT Min                    Free       µs           3.0  4,773,842.9       1,591,281.0
Flow IAT Min                   Free       µs           4.0  1,584,409.6         396,102.4
Fwd IAT Total                  Free       µs         710.0 18,396,494.8          25,910.6
Fwd IAT Max                    Free       µs         632.5 13,509,832.8          21,359.4
Fwd IAT Mean                   Free       µs         375.5  5,262,371.2          14,014.3
min_seg_size_forward           Free    bytes          20.0     81,021.8           4,051.1
Fwd Header Length              Free    bytes          64.0     81,388.4           1,271.7
act_data_pkt_fwd             Costly  packets           1.0        344.4             344.4
Flow IAT Std                   Free       µs      17,985.3  4,433,822.0             246.5
Flow Durat

<!-- STEP C1 -->
**What one step means.** The steps are large compared with an ordinary flow, because the standard
deviations are inflated by outliers (B1). Most are still *possible* for a patient attacker: one step
adds 4.8 s to the shortest gap between the attacker's packets (`Fwd IAT Min`), 18 s to the flow
duration, 13 s to the idle timers, or about 360 extra packets (Costly).

Two Free steps are **not physically possible**:

- `min_seg_size_forward` is a header size, between 0 and 60 bytes for every normal flow, but one step
  adds 81,022 bytes. Its std comes almost entirely from a few recording-bug values around −84 million.
- `Fwd Header Length` (total header bytes sent by the attacker) gains 81,388 bytes per step. That
  would take roughly 2,000 extra packets, which is not free.

We keep the lab's budget exactly as written, so the attack follows the lab. In C2 we check which
features the successful evasions actually used; if they rely on these two, the report says so (and C2
can re-run with each new value capped at the training maximum as a sensitivity check). Rows whose
median is 0 (`Idle …`, `Active …`, `Fwd IAT Std`) show no ratio: most flows never go idle, so any
step is "infinitely" larger than the median.

In [6]:
# STEP C1
groups_table = pd.DataFrame({
    "feature": FEATURES,
    "group": [GROUP[f] for f in FEATURES],
    "rule": [GROUP_RULE[f] for f in FEATURES],
    "step_size": [STEP_SIZE[f] if GROUP[f] != "Fixed" else np.nan for f in FEATURES],
    "train_median": TRAIN_MEDIAN[FEATURES].to_numpy(),
})
groups_table.to_csv("results/tables/C1_feature_groups.csv", index=False, float_format="%.4g")
print("Saved results/tables/C1_feature_groups.csv")

Saved results/tables/C1_feature_groups.csv


<!-- STEP C2 -->
### Step C2: Run the attack, twice

**Greedy search.** Start from one attack flow. At each of five steps, try every change the attacker is
allowed to make, ask the forest for the attack probability of all candidates in **one**
`predict_proba` call, and keep the candidate that lowers the probability most. Stop early when no
candidate helps, or as soon as the flow is classified benign (probability below 0.5): at that point
the attacker has won, and more changes would only inflate the "features changed" count.

| Run | Which features | Direction | Step | Steps |
|---|---|---|---|---|
| **Constrained** | the 30 Free and Costly features, each new value checked with `can_change`, at most 5 different features | up only | +½ training std | 5 |
| **Unconstrained** | all 68 features | up or down | ±½ training std | 5 |
| Constrained + cap (check) | as constrained, but no value may exceed that feature's training maximum | up only | +½ std, capped | 5 |

The unconstrained run keeps the same step size and number of steps, so the **only** difference
between the first two runs is the network constraints. The third run answers C1's question: do the
evasions depend on the physically impossible steps (`min_seg_size_forward`, `Fwd Header Length`)?

In [7]:
# STEP C2
FEATURE_POS = {f: j for j, f in enumerate(FEATURES)}
TRAIN_MAX = X_train.max()


def greedy_attack(model, x_row, constrained=True, steps=5, cap=False):
    """Greedy evasion of one flow.

    x_row: one row as a DataFrame (X_test.iloc[[i]]). It is never modified; every candidate is a copy.
    constrained=True:  Free/Costly features only, increases only (can_change), at most MAX_CHANGES
                       different features. constrained=False: any feature, up or down.
    Both use the same step size (STEP_SIZE, half a training std) and the same number of steps, so the
    only difference between them is the network constraints.
    cap=True (constrained only): a new value may not exceed the feature's training maximum.
    Returns p_before, p_after, changed (one feature name per step taken) and x_adv (the edited row).
    """
    current = x_row.to_numpy(dtype=float)[0].copy()
    p_before = p = float(model.predict_proba(x_row)[0, 1])
    changed = []
    for _ in range(steps):
        if p < 0.5:                                   # already classified benign: the attacker stops
            break
        if constrained:
            budget_left = len(set(changed)) < MAX_CHANGES
            moves = [(f, +1) for f in CHANGEABLE if budget_left or f in changed]
        else:
            moves = [(f, sign) for f in FEATURES for sign in (+1, -1)]

        candidates, names = [], []
        for f, sign in moves:
            j = FEATURE_POS[f]
            new = current[j] + sign * STEP_SIZE[f]
            if constrained:
                if cap:
                    new = min(new, TRAIN_MAX[f])
                if new == current[j] or not can_change(f, current[j], new):
                    continue
            candidate = current.copy()
            candidate[j] = new
            candidates.append(candidate)
            names.append(f)
        if not candidates:
            break
        probs = model.predict_proba(pd.DataFrame(candidates, columns=FEATURES))[:, 1]
        best = int(np.argmin(probs))
        if probs[best] >= p:                          # nothing helps: stop early
            break
        current, p = candidates[best], float(probs[best])
        changed.append(names[best])

    x_adv = pd.DataFrame([current], columns=FEATURES, index=x_row.index)
    return {"p_before": p_before, "p_after": p, "changed": changed, "x_adv": x_adv}

In [8]:
# STEP C2
RUNS = {
    "constrained": dict(constrained=True),
    "unconstrained": dict(constrained=False),
    "constrained + cap": dict(constrained=True, cap=True),
}
X_test_before = X_test.copy()                         # to prove the attack never edits X_test

attack_runs, adv_rows = {}, {}
for run, options in RUNS.items():
    start = time.time()
    rows, advs = [], []
    for pos in ATTACK_IDX:
        result = greedy_attack(rf, X_test.iloc[[pos]], **options)
        rows.append({
            "test_position": int(pos),
            "attack_type": type_test.iloc[pos],
            "p_before": result["p_before"],
            "p_after": result["p_after"],
            "evaded": result["p_after"] < 0.5,
            "steps_taken": len(result["changed"]),
            "n_features_changed": len(set(result["changed"])),
            "changed": ";".join(result["changed"]),
        })
        advs.append(result["x_adv"])
    attack_runs[run] = pd.DataFrame(rows)
    adv_rows[run] = pd.concat(advs)
    print(f"{run:<18} done in {time.time() - start:.0f} s")

assert X_test.equals(X_test_before), "the attack modified X_test"
del X_test_before

# Constraint audit: in the constrained runs, every Fixed column is untouched and nothing went down.
original = X_test.iloc[ATTACK_IDX]
for run in ("constrained", "constrained + cap"):
    diff = adv_rows[run].to_numpy() - original.to_numpy(dtype=float)
    fixed = [FEATURE_POS[f] for f in FEATURES if GROUP[f] == "Fixed"]
    assert np.all(diff[:, fixed] == 0), f"{run}: a Fixed feature changed"
    assert np.all(diff >= 0), f"{run}: a feature decreased"
    if run == "constrained + cap":
        # A raised value never ends above the training maximum (values already above it are untouched).
        raised = diff > 0
        assert np.all(adv_rows[run].to_numpy()[raised] <= np.broadcast_to(TRAIN_MAX.to_numpy(), diff.shape)[raised])
print("Constraint audit passed: no Fixed feature changed and no value decreased in the constrained runs")

constrained        done in 50 s


unconstrained      done in 38 s


constrained + cap  done in 52 s


Constraint audit passed: no Fixed feature changed and no value decreased in the constrained runs


In [9]:
# STEP C2
comparison = pd.DataFrame({
    run: {
        "evaded (of 50)": int(df["evaded"].sum()),
        "mean p before": df["p_before"].mean(),
        "mean p after": df["p_after"].mean(),
        "mean features changed": df["n_features_changed"].mean(),
        "mean steps taken": df["steps_taken"].mean(),
    } for run, df in attack_runs.items()
}).T
comparison.index.name = "run"
comparison.to_csv("results/tables/C2_attack_comparison.csv", float_format="%.4f")

per_flow = pd.concat({run: df for run, df in attack_runs.items()}, names=["run"]).reset_index(level=0)
per_flow.to_csv("results/tables/C2_attack_per_flow.csv", index=False, float_format="%.4f")
comparison.round(4)

,evaded (of 50),mean p before,mean p after,mean features changed,mean steps taken
run,,,,,
constrained,9.0,1.0,0.7311,4.22,4.64
unconstrained,50.0,1.0,0.4312,2.78,2.78
constrained + cap,9.0,1.0,0.7311,4.22,4.64


In [10]:
# STEP C2
# Which attack types evaded, per run.
print(per_flow.pivot_table(index="attack_type", columns="run", values="evaded", aggfunc="sum")
      .assign(flows=type_test.iloc[ATTACK_IDX].value_counts()).to_string())

# Did the successful constrained evasions need the physically impossible header steps (C1 note)?
IMPOSSIBLE = ["min_seg_size_forward", "Fwd Header Length"]
for run in ("constrained", "constrained + cap"):
    ok = attack_runs[run][attack_runs[run]["evaded"]]
    uses = ok["changed"].apply(lambda s: any(f in s.split(";") for f in IMPOSSIBLE)).sum()
    print(f"\n{run}: {len(ok)} evasions, {uses} of them changed {' or '.join(IMPOSSIBLE)}")

# The probability path of the flows the constrained attack did not evade, for the write-up.
c = attack_runs["constrained"]
print("\nConstrained, not evaded: p_after distribution")
print(c.loc[~c["evaded"], "p_after"].describe().round(4).to_string())

run            constrained  constrained + cap  unconstrained  flows
attack_type                                                        
DDoS                     7                  7             25     25
DoS GoldenEye            2                  2              2      2
DoS Hulk                 0                  0             22     22
SSH-Patator              0                  0              1      1

constrained: 9 evasions, 2 of them changed min_seg_size_forward or Fwd Header Length

constrained + cap: 9 evasions, 2 of them changed min_seg_size_forward or Fwd Header Length

Constrained, not evaded: p_after distribution
count    41.0000
mean      0.7921
std       0.0663
min       0.5233
25%       0.7567
50%       0.8200
75%       0.8333
max       0.8667


In [11]:
# STEP C2
# Which features each run used (C3 looks at the constrained run in detail).
for run in ("unconstrained", "constrained"):
    used = attack_runs[run]["changed"].str.split(";").explode().replace("", np.nan).dropna()
    share_fixed = used.map(GROUP).eq("Fixed").mean()
    print(f"{run}: {len(used)} steps, {share_fixed:.0%} of them on Fixed features")
    print(used.value_counts().head(5).rename(lambda f: f"{f} ({GROUP[f]})").to_string(), "\n")

print("Constrained evasions:")
print(attack_runs["constrained"].query("evaded")[["attack_type", "p_after", "changed"]].to_string())

unconstrained: 139 steps, 87% of them on Fixed features
changed
Subflow Bwd Bytes (Fixed)              39
Total Length of Bwd Packets (Fixed)    34
Init_Win_bytes_backward (Fixed)        21
Total Backward Packets (Fixed)         16
Fwd IAT Min (Free)                      6 

constrained: 232 steps, 0% of them on Fixed features
changed
Subflow Fwd Packets (Costly)            41
Total Length of Fwd Packets (Costly)    41
Fwd Packet Length Max (Free)            39
Fwd Header Length (Free)                30
Total Fwd Packets (Costly)              24 

Constrained evasions:
      attack_type   p_after                                                              changed
3   DoS GoldenEye  0.476667  Total Fwd Packets;Fwd Header Length;Subflow Fwd Packets;Fwd IAT Std
11           DDoS  0.423333                  Fwd IAT Min;Flow IAT Min;Fwd IAT Mean;Flow IAT Mean
13  DoS GoldenEye  0.480000  Total Fwd Packets;Subflow Fwd Packets;Fwd Header Length;Fwd IAT Std
15           DDoS  0.416667         

<!-- STEP C2 -->
**What we found.** All 50 flows start at an attack probability of 1.000.

| Run | Evaded (of 50) | Mean p after | Mean features changed |
|---|---|---|---|
| Constrained | **9** | 0.73 | 4.2 |
| Unconstrained | **50** | 0.43 | 2.8 |
| Constrained + cap | 9 | 0.73 | 4.2 |

- **The unconstrained attack evades every flow, with fewer changes**, and 87% of its steps change
  **Fixed** features: the victim's reply bytes (`Subflow Bwd Bytes`, `Total Length of Bwd Packets`)
  and the victim's initial window (`Init_Win_bytes_backward`). It succeeds by editing exactly what a
  real attacker cannot touch, which is why it overstates the risk.
- **The constrained attack evades 9 of 50**: 7 of the 25 DDoS flows and both DoS GoldenEye flows;
  none of the 22 DoS Hulk flows. The seven DDoS evasions needed only 2–4 timing changes, always starting
  with the *shortest* gap between packets: `Fwd IAT Min` (+4.8 s per step) and `Flow IAT Min`
  (+1.6 s per step). Waiting costs the attacker nothing, so this is a realistic evasion. Note, though, that a flood
  that waits seconds between packets in each flow needs many more flows to do the same damage. The
  two GoldenEye evasions used `Fwd Header Length`, one of C1's physically implausible steps.
- **All nine evasions only just cross the line** (final probabilities 0.41–0.48); the 41 flows that
  survived end between 0.52 and 0.87 (mean 0.79). A decision threshold of 0.4 instead of 0.5 would
  have caught all nine, which is worth remembering for E1.
- **Capping at the training maximum changes nothing**: the capped run gives identical results. A
  tree's split thresholds always lie between values seen in training, so any value above the training
  maximum falls on the same side of every split as the maximum itself. For tree models, the
  impossible header steps buy the attacker nothing that a possible step would not.

**Check yourself: which number for a customer, which for the security team?** The gap between 50/50
and 9/50 is protection that comes from the **network** (the attacker cannot rewrite the victim's
replies), not from the model. To a **customer** we would quote the constrained number, 9 of 50 (18%)
of the most obvious attacks evaded by a realistic attacker, because that is the risk they actually
face, stated with its assumptions (five changes, half-std steps, greedy search). To our own **security
team** we would give both: the 50/50 shows that the model itself is not robust at all and leans on
features that only happen to be out of the attacker's reach, so any change in what the attacker can
control (e.g. a compromised server on the other side, or an attacker who controls both ends) would
remove that protection at once. The team should also see *which* features the realistic evasions
used (timing), because that is where to harden the detector.

<!-- STEP C3 -->
### Step C3: See what the attack touched

How often did the **constrained** attack change each feature, over all 50 flows? `attack_counts`
counts steps: each step changes one feature, so a feature raised twice in one flow counts twice. Next
to it we show in how many flows the feature was changed, and how often it was used in the 9 flows that
evaded versus the 41 that did not.

This is also a bug check: the constrained attack may only touch Free and Costly features, so a Fixed
feature anywhere in the counts means `can_change` or `greedy_attack` is wrong.

In [12]:
# STEP C3
constrained = attack_runs["constrained"]
steps_long = (constrained.assign(feature=constrained["changed"].str.split(";"))
              .explode("feature").query("feature != ''").dropna(subset=["feature"]))

attack_counts = steps_long["feature"].value_counts()              # steps per feature, sorted
attack_counts.index.name = "feature"

# The bug check covers every changed feature, not just the top ten.
fixed_used = [f for f in attack_counts.index if GROUP[f] == "Fixed"]
assert not fixed_used, f"the constrained attack changed Fixed features: {fixed_used}"

usage = pd.DataFrame({
    "steps": attack_counts,
    "flows": steps_long.drop_duplicates(["test_position", "feature"])["feature"].value_counts(),
    "steps_in_evaded": steps_long[steps_long["evaded"]]["feature"].value_counts(),
    "steps_in_not_evaded": steps_long[~steps_long["evaded"]]["feature"].value_counts(),
}).fillna(0).astype({"steps": int, "flows": int, "steps_in_evaded": int, "steps_in_not_evaded": int})
# Look the group up by feature name AFTER the table is built: a plain list of labels would be
# matched by position, and pandas reorders the rows when it lines the Series up by feature name.
usage.insert(0, "group", usage.index.map(GROUP))
usage = usage.sort_values(["steps", "flows"], ascending=False)
assert all(usage.loc[f, "group"] == GROUP[f] for f in usage.index)

top10 = usage.head(10)
top10.to_csv("results/tables/C3_attack_top10.csv")
usage.to_csv("results/tables/C3_attack_counts.csv")
print(f"{len(steps_long)} steps over {constrained['test_position'].nunique()} flows, "
      f"{len(attack_counts)} different features, none of them Fixed")
top10

232 steps over 50 flows, 22 different features, none of them Fixed


,group,steps,flows,steps_in_evaded,steps_in_not_evaded
feature,,,,,
Subflow Fwd Packets,Costly,41,41,2,39
Total Length of Fwd Packets,Costly,41,39,0,41
Fwd Packet Length Max,Free,39,20,0,39
Fwd Header Length,Free,30,30,2,28
Total Fwd Packets,Costly,24,24,2,22
Subflow Fwd Bytes,Costly,15,15,0,15
Flow IAT Min,Free,7,7,7,0
Fwd IAT Min,Free,7,7,7,0
Active Std,Free,5,5,0,5


In [13]:
# STEP C3
# Group totals, and the features of the successful evasions only.
print("Steps per group:", steps_long["feature"].map(GROUP).value_counts().to_dict())
evaded_counts = usage[usage["steps_in_evaded"] > 0].sort_values("steps_in_evaded", ascending=False)
print("\nFeatures used by the 9 successful evasions:")
print(evaded_counts[["group", "steps_in_evaded"]].to_string())
timing = [f for f in usage.index if any(k in f for k in ("IAT", "Idle", "Active", "Duration"))]
print(f"\nTiming features: {usage.loc[timing, 'steps'].sum()} of {len(steps_long)} steps overall, "
      f"{usage.loc[timing, 'steps_in_evaded'].sum()} of {usage['steps_in_evaded'].sum()} steps "
      f"in the successful evasions")

Steps per group: {'Costly': 123, 'Free': 109}

Features used by the 9 successful evasions:
                         group  steps_in_evaded
feature                                        
Flow IAT Min              Free                7
Fwd IAT Min               Free                7
Fwd Packet Length Mean    Free                3
Subflow Fwd Packets     Costly                2
Fwd Header Length         Free                2
Total Fwd Packets       Costly                2
Fwd IAT Std               Free                2
Fwd IAT Mean              Free                1
Flow IAT Mean             Free                1

Timing features: 32 of 232 steps overall, 18 of 27 steps in the successful evasions


<!-- STEP C3 -->
**What we found.** The constrained attack took 232 steps over the 50 flows and touched 22 different
features, **none of them Fixed** (checked for every feature, not just the top ten), so the constraint
check works.

| Feature | Group | Steps | Flows | Steps in the 9 evasions |
|---|---|---|---|---|
| Subflow Fwd Packets | Costly | 41 | 41 | 2 |
| Total Length of Fwd Packets | Costly | 41 | 39 | 0 |
| Fwd Packet Length Max | Free | 39 | 20 | 0 |
| Fwd Header Length | Free | 30 | 30 | 2 |
| Total Fwd Packets | Costly | 24 | 24 | 2 |
| Subflow Fwd Bytes | Costly | 15 | 15 | 0 |
| Flow IAT Min | Free | 7 | 7 | 7 |
| Fwd IAT Min | Free | 7 | 7 | 7 |
| Active Std | Free | 5 | 5 | 0 |
| Fwd IAT Std | Free | 3 | 3 | 2 |

The lab expects timing features to lead. **Overall they do not**: the most changed features are the
forward packet and byte counts (Costly, 123 of the 232 steps) and the forward packet sizes. Almost
all of those steps come from the 41 flows that were **not** evaded, mostly DoS Hulk: each extra
packet or padded byte lowers the probability a little, so the greedy search keeps choosing them, but
never enough to cross 0.5.

**Among the 9 successful evasions, timing does lead**: 18 of their 27 steps are timing changes, and
`Flow IAT Min` and `Fwd IAT Min` appear in all 7 DDoS evasions. So the counts show two different
things: what the attacker *tries* (sending more and bigger packets, which mostly fails and makes the
attack louder) and what *works* (waiting longer between packets, which costs nothing). For D1 the
second list is the one to compare with SHAP.